In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root

import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))

    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics


CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}


print(PROJECT_ROOT)

# social media
social_paths = list((CONFIG['model_dir'] / 'multi' ).glob('social_media_*.joblib'))

scaler_path = CONFIG['model_dir'] / 'helper' / 'multi_max_abs_scaler.joblib'
nb_tfidf_path = CONFIG['model_dir'] / 'helper' / 'multi_word_tfidf_vectorizer.joblib'


# gaming
model_dir = CONFIG['model_dir'] / "multi"
gaming_model_paths = list(model_dir.glob("gaming_all_*.joblib"))

# change capture the bert multi



c:\Users\nyuss\OneDrive\Documentos\Bars\Portfollio\Portfollio\Projects\Project-GamingToxicityDetection


In [2]:
np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

# Load Data

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects

In [5]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [6]:
# convert dota labels 
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)


In [7]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = gaming_model_paths
)

In [8]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True)'])

In [9]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_paths,
    scaler_path=scaler_path,
    nb_tfidf_path=nb_tfidf_path,
)

In [11]:
social_media_collection.classifiers

{WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/multi/social_media_multioutput(lr-cv)_pipeline.joblib'): Pipeline(steps=[('clf',
                  MultiOutputClassifier(estimator=LogisticRegressionCV(class_weight='balanced',
                                                                       cv=3,
                                                                       max_iter=1000,
                                                                       random_state=42,
                                                                       scoring='f1_macro',
                                                                       solver='liblinear'),
                                        n_jobs=2))]),
 WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/multi/social_media_multioutput(nb)_pipeline.joblib'): Pipeline(steps=[('clf',
         

In [12]:
bert_multi_collection = BertToxicityModelCollection(
    model_names=["dota_multi", "jigsaw_multi", "wot_multi"]
)

Loading dota_multi from jforward/bert-toxicity/dota_multi_model...
Loading jigsaw_multi from jforward/bert-toxicity/jigsaw_multi_model...
Loading wot_multi from jforward/bert-toxicity/wot_multi_model...


In [13]:
social_gaming_ensemble = Ensemble(
    model_collections=[gaming_model_collection, bert_multi_collection] # social_media_collection --> exclude social media models for multi ensemble
)

# Run Simple Ensemble 

In [13]:
pred_train = social_gaming_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_3class'], pred_train)

Predicting with SimpleMajority...
Input has 53707 samples.
wot_Logistic_Regression: (53707,)
wot_LinearSVC: (53707,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False): (53707,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False): (53707,)
dota_Logistic_Regression: (53707,)
dota_LinearSVC: (53707,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True): (53707,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (53707,)
combined_Logistic_Regression: (53707,)
combined_LinearSVC: (53707,)
combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (53707,)
combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (53707,)
dota_multi: (53707,)
jigsaw_multi: (53707,)
wot_multi: (53707,)
Collected predictions from 15 models.
Prediction matrix shape: (53707, 15)
Aggregating predictions from 15 models...


{'CV Macro F1': 0.913877721098352,
 'CV Weighted F1': 0.9575199344045299,
 'Accuracy': 0.9579943024186791,
 'Coverage': 1.0,
 'Precision': 0.9241171089189614}

In [14]:
pred_val = social_gaming_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_3class'], pred_val)

Predicting with SimpleMajority...
Input has 17056 samples.
wot_Logistic_Regression: (17056,)
wot_LinearSVC: (17056,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False): (17056,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False): (17056,)
dota_Logistic_Regression: (17056,)
dota_LinearSVC: (17056,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True): (17056,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (17056,)
combined_Logistic_Regression: (17056,)
combined_LinearSVC: (17056,)
combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (17056,)
combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (17056,)
dota_multi: (17056,)
jigsaw_multi: (17056,)
wot_multi: (17056,)
Collected predictions from 15 models.
Prediction matrix shape: (17056, 15)
Aggregating predictions from 15 models...


{'CV Macro F1': 0.7210792363819151,
 'CV Weighted F1': 0.8676043301344778,
 'Accuracy': 0.8716580675422139,
 'Coverage': 1.0,
 'Precision': 0.7441405909258364}

# Run Weighted Majority Ensemble 

In [15]:
weights = social_gaming_ensemble.fit_weighted_majority(
    X_val=train[tc],
    y_val=train['label_3class'],
    score_func=metrics.score,
    n_trials=500,
    random_state=CONFIG['seed']
)[0]

In [16]:
weights

array([0.0597675 , 0.14584383, 0.02973528, 0.01812714, 0.00398364,
       0.08179687, 0.10814705, 0.00888044, 0.2013406 , 0.04390604,
       0.00359534, 0.05023792, 0.14648846, 0.04061748, 0.0575324 ])

In [17]:
pred_train = social_gaming_ensemble.predict_weighted_majority(
    X=train[tc],
    weights=weights
)
metrics.metrics(train['label_3class'], pred_train)

{'CV Macro F1': 0.925188987363296,
 'CV Weighted F1': 0.9639875458774941,
 'Accuracy': 0.9639897964883535,
 'Coverage': 1.0,
 'Precision': 0.9253693579968427}

In [18]:
pred_val = social_gaming_ensemble.predict_weighted_majority(
    X=val[tc],
    weights=weights
)
metrics.metrics(val['label_3class'], pred_val)

{'CV Macro F1': 0.7221683838045921,
 'CV Weighted F1': 0.8669211670012367,
 'Accuracy': 0.8696060037523452,
 'Coverage': 1.0,
 'Precision': 0.7363505147704236}

# Run Weighted Confidence Majority Ensemble 

In [ ]:
res = social_gaming_ensemble.fit_weighted_confidence_majority_optuna_cv(
    X_val=train[tc],
    y_val=train['label_3class'],
    score_func=metrics.score,
    n_trials=100,
    random_state=CONFIG['seed']
)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=53707, n_classes=3)
Starting random search for n models: 15
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted pro

In [15]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_3class'], pred_train)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=53707, n_classes=3)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.9173674458077438,
 'CV Weighted F1': 0.9599797016186092,
 'Accuracy': 0.9601914089411063,
 'Coverage': 1.0,
 'Precision': 0.9218595631953234}

In [16]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_3class'], pred_val)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=17056, n_classes=3)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.7150163058475861,
 'CV Weighted F1': 0.8655202385429136,
 'Accuracy': 0.8698991557223265,
 'Coverage': 1.0,
 'Precision': 0.7346280628477054}

In [ ]:
thresholds = np.arange(0.5, 0.9, 0.1)

res = social_gaming_ensemble.fit_weighted_confidence_majority_optuna_cv(
    X_val=train[tc],
    y_val=train['label_3class'],
    score_func=metrics.score,
    n_trials=50,
    random_state=CONFIG['seed'],
    thresholds=thresholds
)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=53707, n_classes=3)
Starting random search for n models: 15
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted probabilities shape: (53707, 3)
Trial weights: (15,)
Weighted pro

In [18]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_3class'], pred_train)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=53707, n_classes=3)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.9136767497479784,
 'CV Weighted F1': 0.9579114574969563,
 'Accuracy': 0.9584225519950844,
 'Coverage': 1.0,
 'Precision': 0.9249056561714403}

In [19]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_3class'], pred_val)

Constructing confidence tensor...
Total models in ensemble: 15
Expected confidence shape: (n_samples=17056, n_classes=3)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.7156052379381546,
 'CV Weighted F1': 0.8655845786100191,
 'Accuracy': 0.8703095684803002,
 'Coverage': 1.0,
 'Precision': 0.7437425093228764}

In [ ]:
res